# SCF migration with reconstructed-space veto

This example keeps only the low-level SCF-specific objects. Model construction, base toy generation and standard plotting use the high-level helpers.


In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
from dalitzplotfitter import (
    DecayChannel,DecayModel,MassWindowVeto,NonResonant,RealImag,SCFSignalPDF,
    SquareDalitzSCFMap,enable_x64,generate_toy,plot_square_dalitz,
)
from dalitzplotfitter.integration import GridIntegrator
enable_x64()


In [ ]:
model=DecayModel(
    DecayChannel("B+",("K+","pi+","pi-")),
    [NonResonant(RealImag(1.0,0.0))],
    normalization_method="square-dalitz",normalization_resolution=140,normalization_pair=(0,2),
)
N=20
nb=N*N
migration=np.eye(nb)*0.70
migration+=np.roll(np.eye(nb),1,axis=1)*0.15
migration+=np.roll(np.eye(nb),-1,axis=1)*0.15
migration/=migration.sum(axis=1,keepdims=True)
fraction=np.linspace(0.08,0.20,nb)
scf=SquareDalitzSCFMap(
    migration,fraction,model.channel.parent_mass,model.channel.daughter_masses,N,N,pair=(0,2)
)
veto=MassWindowVeto((0,2),1.80,1.95)
pdf=SCFSignalPDF(
    intensity=lambda d,p:model.intensity(d,p),
    integrator=GridIntegrator(model.normalization_sample),
    scf_map=scf,veto=veto,
)


In [ ]:
toy=generate_toy(model,20_000,veto=veto,seed=1212)
plot_square_dalitz(
    toy,mother_mass=model.channel.parent_mass,masses=model.channel.daughter_masses,
    pair=(0,2),title="Accepted base signal"
)
plt.show()

density=np.asarray(pdf.numerator(scf.true_bin_data(),{})).reshape(N,N)
plt.figure(figsize=(6,5))
plt.imshow(density.T,origin="lower",aspect="auto")
plt.xlabel("$m'$ bin")
plt.ylabel(r"$\theta'$ bin")
plt.title("SCF + reconstructed veto numerator")
plt.colorbar()
plt.show()
print("normalization:",float(pdf.normalization({})))
